# 00 - LLM Model Server Connection, Clean vs. Noisy Problem Evaluation & Optimum Testing

This diagnostic notebook tests your local LLM server connection, imports system prompts directly from `src/llm/prompts.py`, prompts the LLM to generate continuous optimization algorithms for both **Clean** and **Noisy** problem settings (following LLaMEA's exact internal prompt structure), executes the LLM-generated algorithms on the target BBOB landscape, and compares the returned optimal value against the **True Global Optimum** (`problem.true_optimum`).

## 1. Setup Environment & Initialize LLM Client
Configure path settings and initialize the LLM client provider interface.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import time

# Ensure project root src/ is on sys.path
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Initialize the LLM client for downstream cells in this notebook
from llm.client import get_llm_client
provider = os.environ.get("LLM_PROVIDER", "local")
llm = get_llm_client(provider)

ConnectionError: Could not connect to the local LLM server at 'http://localhost:1234/v1'.
Error details: <urlopen error [Errno 61] Connection refused>
Troubleshooting:
  1. Is your model server running? Start it using: bash scripts/llm_server.sh
  2. Check if the port and URL in your .env are correct: http://localhost:1234/v1
  3. Check if any VPN or proxy is blocking localhost connections.

## 2. Target Problem Setup & True Global Optimum Inspection
Instantiate the target `BBOBProblem` descriptor (Problem ID 1: Sphere, 3D), inspect the **True Global Optimum** (`problem.true_optimum`), and verify how clean and noisy evaluations behave.

In [ ]:
from problems.bbob import BBOBProblem

# Instantiate target problem (BBOB Problem 1: Sphere, 3D)
problem = BBOBProblem(problem_id=1, dim=3)

print("==================== TARGET PROBLEM DESCRIPTOR ====================")
print(f"Problem Name:      {problem.full_meta_str}")
print(f"Search Space Dim:  {problem.dim}D")
print(f"Domain Bounds:     [{problem.lower_bound}, {problem.upper_bound}]")
print(f"True Global Opt:   {problem.true_optimum:.4f}")
print("===================================================================")

# Evaluate candidate point at origin x = [0, 0, 0]
x_origin = np.zeros(problem.dim)
y_clean = problem(x_origin)
y_noisy_dict = problem(x_origin, noise_std=0.05)

print(f"\nEvaluation at origin x = {x_origin}:")
print(f" - Clean Evaluation:  {y_clean:.6f}")
print(f" - Noisy Evaluation:  {y_noisy_dict[0.05]:.6f} (Clean baseline: {y_noisy_dict[0.0]:.6f})")

## 3. Load System Prompts from `src.llm.prompts` & Construct LLaMEA Messages
Import task prompts directly from `src/llm/prompts.py` (`TASK_PROMPT_CLEAN`, `TASK_PROMPT_NOISY`, `EXAMPLE_PROMPT`, and `FORMAT_PROMPT`) and assemble prompt messages exactly as LLaMEA does internally.

In [ ]:
from llm.prompts import TASK_PROMPT_CLEAN, TASK_PROMPT_NOISY, EXAMPLE_PROMPT, FORMAT_PROMPT

# Format Clean Task Prompt using BBOB problem parameters
task_prompt_clean = TASK_PROMPT_CLEAN.format(
    problem_id=problem.problem_id,
    dim=problem.dim,
    lower_bound=problem.lower_bound,
    upper_bound=problem.upper_bound
)

# Format Noisy Task Prompt using BBOB problem parameters
task_prompt_noisy = TASK_PROMPT_NOISY.format(
    problem_id=problem.problem_id,
    dim=problem.dim,
    lower_bound=problem.lower_bound,
    upper_bound=problem.upper_bound
)

# LLaMEA internally constructs initial solution prompts as a single user message containing:
# role_prompt + task_prompt + example_prompt + output_format_prompt
messages_clean = [
    {"role": "user", "content": f"{task_prompt_clean}\n{EXAMPLE_PROMPT}\n{FORMAT_PROMPT}"}
]
messages_noisy = [
    {"role": "user", "content": f"{task_prompt_noisy}\n{EXAMPLE_PROMPT}\n{FORMAT_PROMPT}"}
]

print("==================== LLaMEA EXACT PROMPT MESSAGE PREVIEW ====================")
print(messages_noisy[0]["content"][:450] + "\n... [truncated] ...")
print("=============================================================================")

## 4. Query LLM to Generate Optimization Algorithms (Clean & Noisy)
Query the LLM with the exact LLaMEA prompt message structure to generate optimization algorithm code for Clean and Noisy settings.

In [ ]:
print(f"Querying LLM '{getattr(llm, 'model', 'local')}' for Clean Optimization Algorithm...")
start_t = time.time()
resp_clean = llm.query(messages_clean)
code_clean = llm.extract_algorithm_code(resp_clean)
name_clean = llm.extract_algorithm_description(resp_clean) or "LLM_Clean_Optimizer"
print(f"[OK] Generated Clean Algorithm '{name_clean}' in {time.time() - start_t:.2f}s")

print(f"\nQuerying LLM '{getattr(llm, 'model', 'local')}' for Noisy Optimization Algorithm...")
start_t = time.time()
resp_noisy = llm.query(messages_noisy)
code_noisy = llm.extract_algorithm_code(resp_noisy)
name_noisy = llm.extract_algorithm_description(resp_noisy) or "LLM_Noisy_Optimizer"
print(f"[OK] Generated Noisy Algorithm '{name_noisy}' in {time.time() - start_t:.2f}s")

print("\n==================== FULL GENERATED CLEAN ALGORITHM CODE ====================")
print(code_clean if code_clean else "[No code block extracted]")
print("=============================================================================")

print("\n==================== FULL GENERATED NOISY ALGORITHM CODE ====================")
print(code_noisy if code_noisy else "[No code block extracted]")
print("=============================================================================")

## 5. Execute LLM-Generated Algorithms on Clean & Noisy Problem
Execute the LLM-generated code using `AlgorithmExecutor` on the target problem to retrieve the best returned objective value.

In [ ]:
from core.executor import AlgorithmExecutor

executor = AlgorithmExecutor(timeout_seconds=5.0)
eval_budget = 200

# Helpers for AlgorithmExecutor
def clean_eval(x):
    return float(problem(x))

def noisy_eval(x):
    res = problem(x, noise_std=0.05)
    return res.get(0.05, float('inf')) if isinstance(res, dict) else float(res)

print(f"Executing LLM-Generated Clean Algorithm '{name_clean}'...")
score_clean = executor.execute_algorithm(code_clean, name_clean, problem.dim, clean_eval, eval_budget)

print(f"Executing LLM-Generated Noisy Algorithm '{name_noisy}'...")
score_noisy = executor.execute_algorithm(code_noisy, name_noisy, problem.dim, noisy_eval, eval_budget)

print(f"\n[SUCCESS] Execution complete!")
print(f" - Clean Algorithm Best Value Returned: {score_clean:.4f}")
print(f" - Noisy Algorithm Best Value Returned: {score_noisy:.4f}")

## 6. Compare Returned LLM Optimal Values to Problem True Optimum
Calculate the final absolute error $|f_{\text{returned}} - f^*|$ between the optimal values found by the LLM algorithms and the **True Global Optimum** (`problem.true_optimum`).

In [ ]:
error_clean = abs(score_clean - problem.true_optimum)
error_noisy = abs(score_noisy - problem.true_optimum)

comparison_results = [
    {
        "Evaluation Mode": "Clean (noise_std=0.0)",
        "LLM Algorithm Name": name_clean,
        "Returned Optimal Value": f"{score_clean:.4f}",
        "True Global Optimum": f"{problem.true_optimum:.4f}",
        "Final Error (|Found - True|)": f"{error_clean:.4f}"
    },
    {
        "Evaluation Mode": "Noisy (noise_std=0.05)",
        "LLM Algorithm Name": name_noisy,
        "Returned Optimal Value": f"{score_noisy:.4f}",
        "True Global Optimum": f"{problem.true_optimum:.4f}",
        "Final Error (|Found - True|)": f"{error_noisy:.4f}"
    }
]

df_results = pd.DataFrame(comparison_results)

print(f"==================== LLM OPTIMUM COMPARISON FOR {problem.full_meta_str} ====================")
print(f"Target Problem True Global Optimum: {problem.true_optimum:.4f}")
print("=========================================================================================")
display(df_results) if 'display' in globals() else print(df_results.to_string(index=False))

## 7. Single-Iteration LLaMEA End-to-End Test (Clean & Noisy)
Run 1-iteration tests of full LLaMEA evolution (`run_evolution_for_problem`) for both Clean (`noise_std=0.0`) and Noisy (`noise_std=0.05`) setups to verify full pipeline integration.

In [ ]:
from core.experiment_service import run_experiment
from storage import initialize_storage

print("--- Running Concurrent LLaMEA Evolution (Clean & Noisy) via Orchestrator ---")
storage_manager = initialize_storage("sqlite", root_dir / "data" / "db.sqlite3")

problem_cfg = {
    "problem_id": problem.problem_id,
    "dim": problem.dim,
    "instance_id": problem.instance_id,
}

from functools import partial
from core.experiment_service import ExperimentTask
from core.runner import run_evolution_for_problem

clean_fn = partial(
    run_evolution_for_problem,
    problem=problem,
    llm=llm,
    noise_std=0.0,
    run_id=1,
    checkpoint_dir=root_dir / "data" / "checkpoints",
)
noisy_fn = partial(
    run_evolution_for_problem,
    problem=problem,
    llm=llm,
    noise_std=0.5,
    run_id=1,
    checkpoint_dir=root_dir / "data" / "checkpoints",
)

tasks = [
    ExperimentTask("clean", clean_fn),
    ExperimentTask("noisy", noisy_fn),
]

results = run_experiment(
    tasks=tasks,
    storage_manager=storage_manager,
    checkpoint_dir=root_dir / "data" / "checkpoints"
)

best_clean = results["clean"].best_so_far
best_noisy = results["noisy"].best_so_far

print(f"[OK] Clean evolution complete. Best Solution: {getattr(best_clean, 'name', 'N/A')} | Fitness: {getattr(best_clean, 'fitness', 'N/A')}")
print(f"[OK] Noisy evolution complete. Best Solution: {getattr(best_noisy, 'name', 'N/A')} | Fitness: {getattr(best_noisy, 'fitness', 'N/A')}")
print("[OK] Saved both runs and cleaned up checkpoint files.")